In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


Kaggle credentials set.
Kaggle credentials successfully validated.


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

dlmmdd_workshop_synthetic_source_attribution_challenge_path = kagglehub.competition_download('dlmmdd-workshop-synthetic-source-attribution-challenge')

print('Data source import complete.')


Data source import complete.


In [ ]:
dlmmdd_workshop_synthetic_source_attribution_challenge_path

'/root/.cache/kagglehub/competitions/dlmmdd-workshop-synthetic-source-attribution-challenge'

In [ ]:
!ls /root/.cache/kagglehub/competitions/dlmmdd-workshop-synthetic-source-attribution-challenge

Data  sample_submission.csv


In [ ]:
import copy
import gc
import io
import math
import os
import random
import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageEnhance, ImageFilter, ImageOps
from sklearn.model_selection import StratifiedKFold
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import v2

import timm

In [ ]:
@dataclass
class CFG:
    base_dir: str = "/root/.cache/kagglehub/competitions/dlmmdd-workshop-synthetic-source-attribution-challenge"
    train_csv: str = "/root/.cache/kagglehub/competitions/dlmmdd-workshop-synthetic-source-attribution-challenge/Data/Data/training.csv"
    test_csv: str = "/root/.cache/kagglehub/competitions/dlmmdd-workshop-synthetic-source-attribution-challenge/Data/Data/test.csv"
    submission_csv: str = "submission_convnextv2_large.csv"

    num_classes: int = 10
    seed: int = 20260511
    n_splits: int = 5
    folds: tuple = (0, 1, 2)  
    epochs: int = 17
    batch_size: int = 64
    num_workers: int = 4  # Changed to 0 to prevent Py3.12 multiprocessing DataLoader assertion errors
    persistent_workers: bool = False  # Kaggle Py3.12/PyTorch can print noisy worker __del__ assertions.

    lr_backbone: float = 2e-5
    lr_head: float = 2e-4
    min_lr_ratio: float = 0.02
    warmup_ratio: float = 0.08
    weight_decay: float = 5e-4
    label_smoothing: float = 0.08
    grad_clip_norm: float = 1.0

    ema_decay: float = 0.999
    use_ema: bool = False
    corruption_p: float = 0.90
    tta_modes: tuple = ("orig", "hflip", "center96")

    model_name: str = "convnextv2_large.fcmae_ft_in22k_in1k_384"
    fallback_model_names: tuple = ()
    weight_overlays: dict = None

    # Optional ensemble
    extra_model_names: tuple = ()
    output_dir: str = "dlmmdd_robust_outputs"


cfg = CFG()
cfg.weight_overlays = cfg.weight_overlays or {}
cfg.weight_overlays.setdefault(
    "convnext_tiny.fb_in22k_ft_in1k",
    "/kaggle/input/models/ambrosm/convnext-tiny-fb-in22k-ft-in1k/pytorch/default/1/model.safetensors",
)

'/kaggle/input/models/ambrosm/convnext-tiny-fb-in22k-ft-in1k/pytorch/default/1/model.safetensors'

In [ ]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


seed_everything(cfg.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
Path(cfg.output_dir).mkdir(parents=True, exist_ok=True)
print(f"device={device}  timm={timm.__version__}")

device=cuda  timm=1.0.26


In [ ]:
class HiddenPostProcessAugment:
    """Approximate the undisclosed hidden test processing listed by the host."""

    def __init__(self, p: float = 0.9):
        self.p = p

    def __call__(self, img: Image.Image) -> Image.Image:
        if random.random() > self.p:
            return img
        ops = [
            self.jpeg,
            self.webp,
            self.central_crop_resize,
            self.random_resize,
            self.rotate_preserve_crop,
            self.color_adjust,
            self.blur,
            self.grayscale,
            self.superres_like,
            self.jpeg_ai_like,
        ]
        for op in random.sample(ops, k=random.randint(1, 3)):
            img = op(img)
        return img

    @staticmethod
    def jpeg(img: Image.Image) -> Image.Image:
        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=random.randint(35, 95), optimize=False)
        buf.seek(0)
        return Image.open(buf).convert("RGB")

    @staticmethod
    def webp(img: Image.Image) -> Image.Image:
        buf = io.BytesIO()
        try:
            img.save(buf, format="WEBP", quality=random.randint(35, 95), method=0)
            buf.seek(0)
            return Image.open(buf).convert("RGB")
        except Exception:
            return img

    @staticmethod
    def central_crop_resize(img: Image.Image) -> Image.Image:
        w, h = img.size
        scale = random.uniform(0.78, 0.98)
        nw, nh = max(8, int(w * scale)), max(8, int(h * scale))
        left = (w - nw) // 2
        top = (h - nh) // 2
        return img.crop((left, top, left + nw, top + nh)).resize((w, h), Image.Resampling.BICUBIC)

    @staticmethod
    def random_resize(img: Image.Image) -> Image.Image:
        w, h = img.size
        scale = random.uniform(0.55, 1.25)
        nw, nh = max(32, int(w * scale)), max(32, int(h * scale))
        small = img.resize((nw, nh), random.choice([Image.Resampling.BILINEAR, Image.Resampling.BICUBIC, Image.Resampling.LANCZOS]))
        return small.resize((w, h), Image.Resampling.BICUBIC)

    @staticmethod
    def rotate_preserve_crop(img: Image.Image) -> Image.Image:
        w, h = img.size
        angle = random.uniform(-8.0, 8.0)
        rotated = img.rotate(angle, resample=Image.Resampling.BICUBIC, expand=True)
        return ImageOps.fit(rotated, (w, h), method=Image.Resampling.BICUBIC, centering=(0.5, 0.5))

    @staticmethod
    def color_adjust(img: Image.Image) -> Image.Image:
        img = ImageEnhance.Brightness(img).enhance(random.uniform(0.72, 1.28))
        img = ImageEnhance.Contrast(img).enhance(random.uniform(0.72, 1.35))
        return img

    @staticmethod
    def blur(img: Image.Image) -> Image.Image:
        return img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.25, 1.8)))

    @staticmethod
    def grayscale(img: Image.Image) -> Image.Image:
        return ImageOps.grayscale(img).convert("RGB")

    @staticmethod
    def superres_like(img: Image.Image) -> Image.Image:
        w, h = img.size
        scale = random.uniform(0.45, 0.75)
        low = img.resize((max(16, int(w * scale)), max(16, int(h * scale))), Image.Resampling.BICUBIC)
        up = low.resize((w, h), Image.Resampling.LANCZOS)
        return ImageEnhance.Sharpness(up).enhance(random.uniform(1.2, 2.2))

    @staticmethod
    def jpeg_ai_like(img: Image.Image) -> Image.Image:
        # A cheap proxy for learned compression: remove fine texture, then sharpen.
        img = img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.15, 0.7)))
        return ImageEnhance.Sharpness(img).enhance(random.uniform(0.8, 1.6))


class SquareEvalTransform:
    def __init__(self, image_size: int, mean, std, mode: str = "orig"):
        self.image_size = image_size
        self.mean = mean
        self.std = std
        self.mode = mode
        self.tail = v2.Compose([
            v2.Resize((image_size, image_size), antialias=True),
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=mean, std=std),
        ])

    def __call__(self, img: Image.Image) -> torch.Tensor:
        if self.mode == "hflip":
            img = ImageOps.mirror(img)
        elif self.mode == "center96":
            w, h = img.size
            nw, nh = max(8, int(w * 0.96)), max(8, int(h * 0.96))
            left, top = (w - nw) // 2, (h - nh) // 2
            img = img.crop((left, top, left + nw, top + nh))
        return self.tail(img)


class TrainTransform:
    def __init__(self, image_size: int, mean, std, corruption_p: float):
        self.geom = v2.Compose([
            v2.RandomResizedCrop((image_size, image_size), scale=(0.72, 1.0), ratio=(0.90, 1.10), antialias=True),
            v2.RandomHorizontalFlip(p=0.5),
        ])
        self.hidden_aug = HiddenPostProcessAugment(p=corruption_p)
        self.tail = v2.Compose([
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=mean, std=std),
        ])

    def __call__(self, img: Image.Image) -> torch.Tensor:
        img = self.geom(img)
        img = self.hidden_aug(img)
        return self.tail(img)

In [ ]:
class SIADataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform, is_test: bool):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.is_test = is_test
        self.id_col = next((c for c in ["ID", "id", "image_id"] if c in df.columns), df.columns[0])
        self.path_col = next((c for c in ["path", "image_path", "file_path", "filename"] if c in df.columns), df.columns[1])
        if not is_test:
            self.target_col = "y" if "y" in df.columns else next((c for c in ["TARGET", "target", "label", "class"] if c in df.columns), df.columns[-1])

    def __len__(self) -> int:
        return len(self.df)

    def _resolve_path(self, csv_path: str) -> str:
        csv_path = str(csv_path)
        filename = os.path.basename(csv_path)
        split_dir = "Test" if self.is_test else "Training"
        candidates = []
        if os.path.isabs(csv_path):
            candidates.append(csv_path)
        candidates.extend([
            os.path.join(cfg.base_dir, csv_path),
            os.path.join(cfg.base_dir, "Data", csv_path),
            os.path.join(cfg.base_dir, "Data", "Data", csv_path),
            os.path.join(cfg.base_dir, "Data", "Data", split_dir, filename),
            os.path.join(cfg.base_dir, "Data", split_dir, filename),
            os.path.join(cfg.base_dir, split_dir, filename),
        ])
        for path in candidates:
            if os.path.exists(path):
                return path
        return candidates[0]

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(self._resolve_path(row[self.path_col])).convert("RGB")
        img = self.transform(img)
        if self.is_test:
            return img, str(row[self.id_col])
        return img, int(row[self.target_col])

In [ ]:
def create_model_with_fallback(model_name: str):
    names = (model_name,) + tuple(cfg.fallback_model_names)
    last_error = None
    for name in names:
        try:
            overlay = {}
            if name in cfg.weight_overlays and os.path.exists(cfg.weight_overlays[name]):
                overlay = {"file": cfg.weight_overlays[name]}
            elif name in cfg.weight_overlays:
                print(f"configured local weight not found for {name}: {cfg.weight_overlays[name]}")
            kwargs = {"pretrained": True, "num_classes": cfg.num_classes}
            if overlay:
                kwargs["pretrained_cfg_overlay"] = overlay
            model = timm.create_model(name, **kwargs)
            print(f"using model: {name}")
            return model, name
        except Exception as exc:
            print(f"could not create {name}: {exc}")
            last_error = exc
    raise RuntimeError(f"No configured model could be created. Last error: {last_error}")


def split_head_backbone_params(model: nn.Module):
    head_prefixes = ("head", "classifier", "fc")
    head, backbone = [], []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if name.startswith(head_prefixes) or ".head." in name or ".classifier." in name or ".fc." in name:
            head.append(param)
        else:
            backbone.append(param)
    return [
        {"params": backbone, "lr": cfg.lr_backbone},
        {"params": head, "lr": cfg.lr_head},
    ]


class ModelEMA:
    def __init__(self, model: nn.Module, decay: float):
        self.decay = decay
        self.module = copy.deepcopy(model).eval()
        for p in self.module.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model: nn.Module):
        msd = model.state_dict()
        for key, ema_v in self.module.state_dict().items():
            model_v = msd[key].detach()
            if ema_v.dtype.is_floating_point:
                ema_v.mul_(self.decay).add_(model_v, alpha=1.0 - self.decay)
            else:
                ema_v.copy_(model_v)


def make_scheduler(optimizer, total_steps: int):
    warmup_steps = max(1, int(total_steps * cfg.warmup_ratio))

    def lr_lambda(step):
        if step < warmup_steps:
            return float(step + 1) / warmup_steps
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
        return cfg.min_lr_ratio + (1.0 - cfg.min_lr_ratio) * cosine

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)


def accuracy_from_logits(logits, targets) -> float:
    return (logits.argmax(1) == targets).float().mean().item()

In [ ]:
def train_one_fold(model_name: str, train_df: pd.DataFrame, test_df: pd.DataFrame, fold: int, train_idx, val_idx):
    model, resolved_name = create_model_with_fallback(model_name)
    try:
        data_cfg = timm.resolve_model_data_config(model)
    except AttributeError:
        data_cfg = timm.resolve_data_config(model.pretrained_cfg)
    image_size = int(data_cfg["input_size"][-1])
    mean, std = data_cfg["mean"], data_cfg["std"]
    print(f"fold={fold} image_size={image_size} mean={mean} std={std}")

    train_ds = SIADataset(train_df.iloc[train_idx], TrainTransform(image_size, mean, std, cfg.corruption_p), is_test=False)
    val_ds = SIADataset(train_df.iloc[val_idx], SquareEvalTransform(image_size, mean, std, "orig"), is_test=False)

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=device.type == "cuda",
        drop_last=True,
        persistent_workers=cfg.persistent_workers and cfg.num_workers > 0,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=cfg.batch_size * 2,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=device.type == "cuda",
        persistent_workers=cfg.persistent_workers and cfg.num_workers > 0,
    )

    model = model.to(device, memory_format=torch.channels_last)
    ema = ModelEMA(model, cfg.ema_decay) if cfg.use_ema else None
    criterion = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)
    optimizer = torch.optim.AdamW(split_head_backbone_params(model), weight_decay=cfg.weight_decay)
    total_steps = cfg.epochs * len(train_loader)
    scheduler = make_scheduler(optimizer, total_steps)
    try:
        scaler = torch.amp.GradScaler(device.type, enabled=device.type == "cuda")
    except TypeError:
        scaler = torch.cuda.amp.GradScaler(enabled=device.type == "cuda")

    best_acc = -1.0
    best_loss = float("inf")
    best_path = Path(cfg.output_dir) / f"{resolved_name.replace('/', '_')}_fold{fold}.pth"
    global_step = 0

    for epoch in range(cfg.epochs):
        model.train()
        t0 = time.time()
        running_loss, running_acc, seen = 0.0, 0.0, 0
        pbar = tqdm(train_loader, desc=f"{resolved_name} fold {fold} epoch {epoch + 1}/{cfg.epochs}")
        for images, targets in pbar:
            images = images.to(device, non_blocking=True, memory_format=torch.channels_last)
            targets = targets.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
                logits = model(images)
                loss = criterion(logits, targets)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip_norm)
            old_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            new_scale = scaler.get_scale()
            if new_scale >= old_scale:
                scheduler.step()
            global_step += 1
            if ema is not None:
                ema.update(model)

            bs = images.size(0)
            seen += bs
            running_loss += loss.item() * bs
            running_acc += accuracy_from_logits(logits.detach(), targets) * bs
            pbar.set_postfix(loss=running_loss / seen, acc=running_acc / seen, lr=optimizer.param_groups[0]["lr"])

        val_acc, val_loss = validate(model, val_loader, criterion)
        elapsed = (time.time() - t0) / 60
        print(f"epoch={epoch + 1} val_loss={val_loss:.4f} val_acc={val_acc:.5f} minutes={elapsed:.1f}")
        improved = (val_acc > best_acc) or (val_acc == best_acc and val_loss < best_loss)
        if improved:
            best_acc = val_acc
            best_loss = val_loss
            torch.save(
                {
                    "model": (ema.module if ema is not None else model).state_dict(),
                    "model_name": resolved_name,
                    "image_size": image_size,
                    "mean": mean,
                    "std": std,
                    "fold": fold,
                    "val_acc": best_acc,
                    "val_loss": best_loss,
                },
                best_path,
            )
            print(f"saved {best_path} val_acc={best_acc:.5f} val_loss={best_loss:.5f}")

    del model, ema, optimizer, scheduler, scaler, train_loader, val_loader
    gc.collect()
    torch.cuda.empty_cache()
    return best_path, best_acc, best_loss


@torch.no_grad()
def validate(model: nn.Module, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for images, targets in tqdm(loader, desc="valid", leave=False):
        images = images.to(device, non_blocking=True, memory_format=torch.channels_last)
        targets = targets.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
            logits = model(images)
            loss = criterion(logits, targets)
        total_loss += loss.item() * images.size(0)
        correct += (logits.argmax(1) == targets).sum().item()
        total += targets.numel()
    return correct / total, total_loss / total

In [ ]:
@torch.no_grad()
def predict_checkpoint(checkpoint_path: Path, test_df: pd.DataFrame) -> np.ndarray:
    try:
        ckpt = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    except TypeError:
        ckpt = torch.load(checkpoint_path, map_location="cpu")
    model = timm.create_model(ckpt["model_name"], pretrained=False, num_classes=cfg.num_classes)
    model.load_state_dict(ckpt["model"])
    model = model.to(device, memory_format=torch.channels_last).eval()

    all_tta_logits = []
    for mode in cfg.tta_modes:
        transform = SquareEvalTransform(ckpt["image_size"], ckpt["mean"], ckpt["std"], mode)
        ds = SIADataset(test_df, transform, is_test=True)
        loader = DataLoader(
            ds,
            batch_size=cfg.batch_size * 2,
            shuffle=False,
            num_workers=cfg.num_workers,
            pin_memory=device.type == "cuda",
            persistent_workers=cfg.persistent_workers and cfg.num_workers > 0,
        )
        logits_list, ids = [], []
        for images, batch_ids in tqdm(loader, desc=f"predict {checkpoint_path.name} {mode}"):
            images = images.to(device, non_blocking=True, memory_format=torch.channels_last)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
                logits = model(images)
            logits_list.append(logits.float().cpu().numpy())
            ids.extend(list(batch_ids))
        assert list(map(str, test_df[ds.id_col])) == ids
        all_tta_logits.append(np.concatenate(logits_list, axis=0))
        del loader, ds

    del model
    gc.collect()
    torch.cuda.empty_cache()
    return np.mean(all_tta_logits, axis=0)

In [ ]:
import timm.data
timm.resolve_model_data_config = timm.data.resolve_model_data_config
timm.resolve_data_config = timm.data.resolve_data_config

def run_training_and_submission():
    train_df = pd.read_csv(cfg.train_csv)
    test_df = pd.read_csv(cfg.test_csv)
    print(train_df.head())
    print(test_df.head())
    target_col = "y" if "y" in train_df.columns else "TARGET"

    model_names = (cfg.model_name,) + tuple(cfg.extra_model_names)
    checkpoints = []
    score_rows = []
    for model_name in model_names:
        kf = StratifiedKFold(n_splits=cfg.n_splits, shuffle=True, random_state=cfg.seed)
        for fold, (train_idx, val_idx) in enumerate(kf.split(train_df, train_df[target_col])):
            if fold not in cfg.folds:
                continue
            ckpt_path, best_acc, best_loss = train_one_fold(model_name, train_df, test_df, fold, train_idx, val_idx)
            checkpoints.append(ckpt_path)
            score_rows.append({
                "model": model_name,
                "fold": fold,
                "val_acc": best_acc,
                "val_loss": best_loss,
                "checkpoint": str(ckpt_path),
            })

    pd.DataFrame(score_rows).to_csv(Path(cfg.output_dir) / "fold_scores.csv", index=False)
    print(pd.DataFrame(score_rows))

    logits = []
    for ckpt_path in checkpoints:
        logits.append(predict_checkpoint(ckpt_path, test_df))
    mean_logits = np.mean(logits, axis=0)
    preds = mean_logits.argmax(1)

    id_col = next((c for c in ["ID", "id", "image_id"] if c in test_df.columns), test_df.columns[0])
    sub = pd.DataFrame({"ID": test_df[id_col], "TARGET": preds.astype(int)})
    sub.to_csv(cfg.submission_csv, index=False)
    print(sub.head())
    print(f"saved {cfg.submission_csv}")


if __name__ == "__main__":
    run_training_and_submission()

   ID                 path  y
0   0  Data/Training/0.png  0
1   1  Data/Training/1.png  8
2   2  Data/Training/2.png  5
3   3  Data/Training/3.png  0
4   4  Data/Training/4.png  9
   ID              path
0   6   Data/Test/6.png
1  12  Data/Test/12.png
2  16  Data/Test/16.png
3  17  Data/Test/17.png
4  18  Data/Test/18.png
using model: convnextv2_large.fcmae_ft_in22k_in1k_384
fold=0 image_size=384 mean=(0.485, 0.456, 0.406) std=(0.229, 0.224, 0.225)


convnextv2_large.fcmae_ft_in22k_in1k_384 fold 0 epoch 1/17:   0%|          | 0/87 [00:00<?, ?it/s]

valid:   0%|          | 0/11 [00:00<?, ?it/s]

epoch=1 val_loss=1.5077 val_acc=0.61357 minutes=2.8
saved dlmmdd_robust_outputs/convnextv2_large.fcmae_ft_in22k_in1k_384_fold0.pth val_acc=0.61357 val_loss=1.50767


convnextv2_large.fcmae_ft_in22k_in1k_384 fold 0 epoch 2/17:   0%|          | 0/87 [00:00<?, ?it/s]

...

predict convnextv2_large.fcmae_ft_in22k_in1k_384_fold2.pth hflip:   0%|          | 0/24 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ede4900b9c0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ede4900b9c0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

predict convnextv2_large.fcmae_ft_in22k_in1k_384_fold2.pth center96:   0%|          | 0/24 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ede4900b9c0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^Exception ignored in: ^^<function _MultiProcessingDataLoaderIter.__del__ at 0x7ede4900b9c0>^^
^Traceback (most recent call last):
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
^^    ^self._shutdown_workers()
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^    ^if w.is_alive():^
^ ^  ^ ^^ ^ ^ 

   ID  TARGET
0   6       6
1  12       1
2  16       7
3  17       1
4  18       6
saved submission_convnextv2_large.csv


In [ ]:
!nvidia-smi

Tue May 12 02:06:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   39C    P0             64W /  400W |    1134MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

output_folder = '/content/dlmmdd_robust_outputs'
zip_file_name = 'dlmmdd_robust_outputs.zip'
zip_path = os.path.join('/content', zip_file_name)

# Zip the folder
!zip -r {zip_path} {output_folder}

drive_path = '/content/drive/My Drive/'
!cp {zip_path} "{drive_path}"

print(f"Folder '{output_folder}' zipped to '{zip_path}' and copied to '{drive_path}'")

  adding: content/dlmmdd_robust_outputs/ (stored 0%)
  adding: content/dlmmdd_robust_outputs/convnextv2_large.fcmae_ft_in22k_in1k_384_fold0.pth (deflated 7%)
  adding: content/dlmmdd_robust_outputs/convnextv2_large.fcmae_ft_in22k_in1k_384_fold1.pth (deflated 7%)
  adding: content/dlmmdd_robust_outputs/fold_scores.csv (deflated 65%)
  adding: content/dlmmdd_robust_outputs/convnextv2_large.fcmae_ft_in22k_in1k_384_fold2.pth (deflated 7%)
Folder '/content/dlmmdd_robust_outputs' zipped to '/content/dlmmdd_robust_outputs.zip' and copied to '/content/drive/My Drive/'
